# Morgan Interaction Training

Ordered interaction Morgan baseline setup and training. Uses the fixed data splits and calibration meta-fit/meta-select partition. Set the recorded environment before execution.


## Stage 1

Source: `side_effect_and_baseline_history`, cell 77 (zero-based).


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path("/content/drive/MyDrive/DDI_V2")

train_path = ROOT / "clean_train_split.csv"
cal_path = ROOT / "clean_calibration_split.csv"

for path in [train_path, cal_path]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

train = pd.read_csv(train_path)
cal = pd.read_csv(cal_path)

required = ["Drug1", "Drug2", "Label", "label_id"]

for name, df in [("Train", train), ("Calibration", cal)]:
    missing = set(required) - set(df.columns)
    assert not missing, f"{name}: missing columns {missing}"
    assert not df[required].isna().any().any(), f"{name}: missing values"

    numeric_ids = pd.to_numeric(df["label_id"], errors="raise")
    assert (numeric_ids % 1 == 0).all(), f"{name}: non-integer labels"
    df["label_id"] = numeric_ids.astype(int)

    assert df["label_id"].between(0, 83).all()
    print(f"{name}: {len(df):,} rows | {df['label_id'].nunique()} classes")

assert set(train["label_id"]) == set(range(84)), \
    "Train does not contain all 84 classes."


mapping = pd.concat(
    [train[["label_id", "Label"]], cal[["label_id", "Label"]]],
    ignore_index=True
).drop_duplicates()

assert mapping.groupby("label_id")["Label"].nunique().max() == 1, \
    "One label_id maps to different labels."

assert mapping.groupby("Label")["label_id"].nunique().max() == 1, \
    "One label maps to different IDs."



def record_keys(df):
    return pd.MultiIndex.from_frame(
        df[["Drug1", "Drug2", "label_id"]]
    )

overlap = record_keys(train).intersection(record_keys(cal))

print("Exact Train/Calibration record overlap:", len(overlap))
assert len(overlap) == 0, \
    "Split overlap detected; inspect partitioning before training."


cal_grouped = cal[
    ["Drug1", "Drug2", "label_id"]
].drop_duplicates().reset_index(drop=True)

counts = cal_grouped["label_id"].value_counts()



rare_mask = cal_grouped["label_id"].map(counts).lt(2)
rare_group_ids = cal_grouped.index[rare_mask].to_numpy()
regular_group_ids = cal_grouped.index[~rare_mask].to_numpy()

assert len(regular_group_ids) > 0, "No classes suitable for internal splitting."

fit_groups, select_groups = train_test_split(
    regular_group_ids,
    test_size=0.25,
    random_state=42,
    stratify=cal_grouped.loc[regular_group_ids, "label_id"],
)

fit_groups = np.concatenate([fit_groups, rare_group_ids])

select_keys = record_keys(cal_grouped.iloc[select_groups])
is_select = record_keys(cal).isin(select_keys)

meta_fit_idx = np.flatnonzero(~is_select)
meta_select_idx = np.flatnonzero(is_select)

assert len(np.intersect1d(meta_fit_idx, meta_select_idx)) == 0
assert len(meta_fit_idx) + len(meta_select_idx) == len(cal)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = ROOT / f"COMMENT4_REBUILD_{stamp}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

np.save(RUN_DIR / "meta_fit_calibration_indices.npy", meta_fit_idx)
np.save(RUN_DIR / "meta_select_calibration_indices.npy", meta_select_idx)

mapping.sort_values("label_id").to_csv(
    RUN_DIR / "label_mapping.csv", index=False
)

manifest = pd.DataFrame({
    "calibration_row_index": np.arange(len(cal)),
    "role": np.where(is_select, "meta_select", "meta_fit"),
    "label_id": cal["label_id"].to_numpy(),
})
manifest.to_csv(RUN_DIR / "calibration_roles.csv", index=False)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

protocol = {
    "status": "development_split_prepared_not_trained",
    "train_path": str(train_path),
    "calibration_path": str(cal_path),
    "train_sha256": sha256(train_path),
    "calibration_sha256": sha256(cal_path),
    "train_rows": len(train),
    "calibration_rows": len(cal),
    "meta_fit_rows": len(meta_fit_idx),
    "meta_select_rows": len(meta_select_idx),
    "internal_split_seed": 42,
    "planned_training_seeds": [42, 43, 44],
    "singleton_classes_fit_only": [
        int(x) for x in counts[counts < 2].index
    ],
    "test_used_in_this_step": False,
    "historical_test_previously_used_for_selection": True,
    "note": (
        "Exact-record overlap checked. This does not establish "
        "drug-disjoint or scaffold-disjoint evaluation."
    ),
}

(RUN_DIR / "protocol.json").write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8"
)

print("\nMeta-fit:", len(meta_fit_idx))
print("Meta-selection:", len(meta_select_idx))
print("Classes confined to meta-fit:", protocol["singleton_classes_fit_only"])
print("\nSaved:", RUN_DIR)
print("\nInputs prepared for feature construction and base-model training.")

## Stage 2

Source: `side_effect_and_baseline_history`, cell 78 (zero-based).


In [ ]:
%pip -q install lightgbm rdkit

## Stage 3

Source: `side_effect_and_baseline_history`, cell 79 (zero-based).


In [ ]:
from pathlib import Path
import hashlib
import json
import time
import platform

import numpy as np
import pandas as pd
import scipy
from scipy.sparse import csr_matrix, hstack
import sklearn
from sklearn.utils.class_weight import compute_sample_weight

import lightgbm as lgb
import rdkit
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

ROOT = Path("/content/drive/MyDrive/DDI_V2")
RUN = ROOT / "COMMENT4_REBUILD_20260925_134439"
OUT = RUN / "MORGAN_3SEEDS"
OUT.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 43, 44]

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

protocol = json.loads((RUN / "protocol.json").read_text())


assert file_hash(ROOT / "clean_train_split.csv") == protocol["train_sha256"]
assert file_hash(ROOT / "clean_calibration_split.csv") == protocol["calibration_sha256"]

train = pd.read_csv(ROOT / "clean_train_split.csv")
cal = pd.read_csv(ROOT / "clean_calibration_split.csv")

y_train = train["label_id"].to_numpy(dtype=np.int64)
y_cal = cal["label_id"].to_numpy(dtype=np.int64)

fit_idx = np.load(RUN / "meta_fit_calibration_indices.npy")
select_idx = np.load(RUN / "meta_select_calibration_indices.npy")

assert set(y_train) == set(range(84))
assert len(np.intersect1d(fit_idx, select_idx)) == 0
assert np.array_equal(
    np.sort(np.concatenate([fit_idx, select_idx])),
    np.arange(len(cal))
)


smiles = pd.unique(pd.concat([
    train["Drug1"], train["Drug2"],
    cal["Drug1"], cal["Drug2"]
], ignore_index=True).astype(str))

generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048,
    includeChirality=False,
)

fp_rows, fp_cols = [], []
invalid = []

for i, smi in enumerate(smiles):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        invalid.append(smi)
        continue

    bits = list(generator.GetFingerprint(mol).GetOnBits())
    fp_rows.extend([i] * len(bits))
    fp_cols.extend(bits)


if invalid:
    pd.DataFrame({"invalid_smiles": invalid}).to_csv(
        OUT / "invalid_smiles.csv", index=False
    )
    raise ValueError(
        f"Stopped: {len(invalid)} invalid SMILES. "
        "Resolve the validation error before continuing training."
    )

drug_fp = csr_matrix(
    (
        np.ones(len(fp_rows), dtype=np.float32),
        (fp_rows, fp_cols)
    ),
    shape=(len(smiles), 2048),
)

drug_to_id = {s: i for i, s in enumerate(smiles)}

def pair_features(df):
    ids1 = df["Drug1"].astype(str).map(drug_to_id)
    ids2 = df["Drug2"].astype(str).map(drug_to_id)

    assert ids1.notna().all() and ids2.notna().all()

    x1 = drug_fp[ids1.to_numpy(dtype=np.int64)]
    x2 = drug_fp[ids2.to_numpy(dtype=np.int64)]
    common = x1.multiply(x2)

    only1 = x1 - common
    only2 = x2 - common
    only1.eliminate_zeros()
    only2.eliminate_zeros()

    n1 = np.asarray(x1.sum(axis=1)).ravel()
    n2 = np.asarray(x2.sum(axis=1)).ravel()
    nc = np.asarray(common.sum(axis=1)).ravel()
    union = n1 + n2 - nc
    eps = 1e-8

    scalars = np.column_stack([
        n1, n2, nc, union,
        nc / (union + eps),
        nc / (n1 + eps),
        nc / (n2 + eps),
        n1 - n2,
        np.abs(n1 - n2),
    ]).astype(np.float32)

    return hstack(
        [x1, x2, common, only1, only2, csr_matrix(scalars)],
        format="csr",
        dtype=np.float32
    )

print("Building features...", flush=True)
X_train = pair_features(train)
X_cal = pair_features(cal)

assert X_train.shape == (len(train), 10249)
assert X_cal.shape == (len(cal), 10249)

print("Train:", X_train.shape)
print("Calibration:", X_cal.shape)

weights = compute_sample_weight("balanced", y_train)

params = {
    "objective": "multiclass",
    "num_class": 84,
    "n_estimators": 1000,
    "learning_rate": 0.04,
    "num_leaves": 95,
    "max_depth": -1,
    "min_child_samples": 25,
    "subsample": 0.85,
    
    
    "subsample_freq": 0,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "n_jobs": 2,
    "verbosity": -1,
    "deterministic": True,
    "force_col_wise": True,
}

config = {
    "model": "Morgan LightGBM — new controlled run",
    "parameters": params,
    "seeds": SEEDS,
    "features": {
        "radius": 2,
        "bits": 2048,
        "includeChirality": False,
        "pair_dimension": 10249,
        "blocks": [
            "Drug1", "Drug2", "common",
            "Drug1_only", "Drug2_only", "9_scalars"
        ],
    },
    "sample_weight": "balanced, computed from Train only",
    "early_stopping": {
        "rounds": 70,
        "metric": "multi_logloss",
        "partition": "meta_fit subset of Calibration",
    },
    "meta_selection_used_here": False,
    "test_used_here": False,
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "sklearn": sklearn.__version__,
        "lightgbm": lgb.__version__,
        "rdkit": rdkit.__version__,
    },
    "train_sha256": protocol["train_sha256"],
    "calibration_sha256": protocol["calibration_sha256"],
    "meta_fit_indices_sha256": file_hash(
        RUN / "meta_fit_calibration_indices.npy"
    ),
}

config_text = json.dumps(config, indent=2, sort_keys=True)
config_path = OUT / "config.json"

if config_path.exists():
    assert config_path.read_text() == config_text, (
        "Configuration or package versions changed; "
        "do not resume a run with different settings."
    )
else:
    config_path.write_text(config_text)


cal.to_csv(OUT / "calibration_rows.csv", index=False)
np.save(OUT / "class_order.npy", np.arange(84))

summaries = []

for seed in SEEDS:
    seed_dir = OUT / f"seed_{seed}"
    seed_dir.mkdir(exist_ok=True)
    done_path = seed_dir / "completed.json"

    if done_path.exists():
        for filename in ["model.txt", "calibration_probabilities.npy",
                         "training_history.json"]:
            assert (seed_dir / filename).exists(), filename
        summaries.append(json.loads(done_path.read_text()))
        print(f"Seed {seed}: already completed — skipped.")
        continue

    print(f"\nTRAINING SEED {seed}", flush=True)
    started = time.time()

    model = lgb.LGBMClassifier(
        **params,
        random_state=seed
    )

    history = {}
    model.fit(
        X_train,
        y_train,
        sample_weight=weights,
        eval_set=[(X_cal[fit_idx], y_cal[fit_idx])],
        eval_metric="multi_logloss",
        callbacks=[
            lgb.early_stopping(70, first_metric_only=True),
            lgb.log_evaluation(25),
            lgb.record_evaluation(history),
        ],
    )

    assert np.array_equal(model.classes_, np.arange(84))

    probabilities = model.predict_proba(
        X_cal,
        num_iteration=model.best_iteration_
    ).astype(np.float32)

    assert probabilities.shape == (len(cal), 84)
    assert np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)

    model.booster_.save_model(
        str(seed_dir / "model.txt"),
        num_iteration=model.best_iteration_
    )
    np.save(
        seed_dir / "calibration_probabilities.npy",
        probabilities
    )
    (seed_dir / "training_history.json").write_text(
        json.dumps(history, indent=2)
    )

    result = {
        "seed": seed,
        "best_iteration": int(model.best_iteration_),
        "early_stopping_logloss": float(
            model.best_score_["valid_0"]["multi_logloss"]
        ),
        "minutes": round((time.time() - started) / 60, 2),
        "test_evaluated": False,
    }

    
    done_path.write_text(json.dumps(result, indent=2))
    summaries.append(result)
    print(result, flush=True)

pd.DataFrame(summaries).to_csv(
    OUT / "training_summary.csv", index=False
)

print("\nCompleted Morgan training:")
display(pd.DataFrame(summaries))
print("Saved:", OUT)
print("Test has not been evaluated.")